# 阶段三：PPF 粗位姿估计与 ICP 位姿细化

## 方法目标

将几何路线中的两个环节接起来：先用 PPF 从 CAD 与观测点云产生粗位姿候选，再用 ICP 从候选初值开始做局部细化。

流程如下：

1. 用 BOP 实例可见掩码筛选深度点，并反投影到相机坐标系。
2. 从 CAD 网格采样模型点云。
3. 对模型和观测点云计算法线；用 PPF 特征匹配和投票估计物体到相机的候选位姿 $T_{CO}$。
4. 将 PPF 候选的逆变换作为 ICP 初值，把观测点云对齐到 CAD 点云。
5. 比较 PPF 粗位姿和 ICP 细化结果，并显示点云对齐情况。
6. 另外用扰动真值构造一个较好的初值，作为 ICP 局部细化的对照。

本实验使用 BOP 真值可见掩码提供目标区域，因此不评估目标检测或分割。PPF 位姿估计不使用真值位姿；真值只用于计算误差和构造单独的对照初值。


## 坐标方向

BOP 的真值变换将 CAD 模型点从物体坐标系 O 变换到相机坐标系 C：

$$
\mathbf p_C = R_{CO}\mathbf p_O + \mathbf t_{CO}.
$$

ICP 将深度观测点云从相机坐标系 C 对齐到 CAD 所在的物体坐标系 O，因此传入 ICP 的初始变换是 $T_{CO}^{-1}$。ICP 返回 $T_{CO}$ 的估计值后，再求逆得到物体到相机的位姿估计。

单位采用毫米。ICP 的对应距离阈值、点云坐标、位移扰动和误差输出均使用毫米。


## 数据与依赖

数据位置沿用前两个练习：

~~~text
.local/data/bop/ycbv/
├── models/obj_XXXXXX.ply
└── test/<scene_id>/
    ├── depth/
    ├── mask_visib/
    ├── scene_camera.json
    ├── scene_gt.json
    └── scene_gt_info.json
~~~

需要 NumPy、Open3D 和 Plotly。默认选择可见比例较高的一个实例；可用 `SCENE_ID` 限定场景范围。


In [10]:
import json
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "6D位姿估计":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / ".local" / "data" / "bop" / "ycbv"
SPLIT_DIR = DATASET_ROOT / "test"
MODELS_DIR = DATASET_ROOT / "models"

# None 表示扫描全部 test 场景；整数表示只使用指定场景。
SCENE_ID = None

# 公共点云采样参数，长度单位为毫米。
CAD_POINT_COUNT = 20000
VOXEL_SIZE_MM = 3.0

# PPF 粗位姿估计参数；点数上限用于控制 O(N^2) 的点对计算。
PPF_MODEL_POINT_COUNT = 260         # CAD 模型点云的采样点数
PPF_SCENE_POINT_COUNT = 220         # 场景点云的采样点数
PPF_SCENE_REFERENCE_COUNT = 120     # 参与 PPF 投票的场景参考点数
PPF_DISTANCE_STEP_MM = 8.0          # 点对距离特征的量化步长
PPF_ANGLE_STEP_DEG = 18.0           # 点对法向夹角特征的量化步长
PPF_MAX_MATCHES_PER_FEATURE = 40    # 每个 PPF 特征保留的模型匹配数上限
PPF_TOP_VOTES_PER_REFERENCE = 3     # 每个场景参考点保留的高票候选数
PPF_CLUSTER_TRANSLATION_MM = 25.0   # 位姿聚类的平移阈值
PPF_CLUSTER_ROTATION_DEG = 22.0     # 位姿聚类的旋转阈值
PPF_NORMAL_RADIUS_MM = 16.0         # 估计表面法向量时的邻域半径

# ICP 局部细化参数。
MAX_CORRESPONDENCE_DISTANCE_MM = 20.0   # ICP 对应点的最大距离
MAX_ICP_ITERATIONS = 80                 # ICP 的最大迭代次数

# 对照实验参数：通过扰动真值构造 ICP 初值。
PERTURB_ROTATION_DEG = 5.0
PERTURB_TRANSLATION_MM = np.array([6.0, -4.0, 5.0])


def read_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


## 1. 样本选择

同一帧可能包含多个实例。代码读取 `scene_gt_info.json` 中的 `visib_fract`，并在选定场景范围内选择可见比例最高的实例。输出包括场景、图像、实例序号和模型编号。


In [11]:
scene_dirs = sorted(
    path for path in SPLIT_DIR.iterdir()
    if path.is_dir() and path.name.isdigit()
)
if SCENE_ID is not None:
    scene_dirs = [path for path in scene_dirs if path.name == f"{SCENE_ID:06d}"]
if not scene_dirs:
    raise FileNotFoundError(f"没有找到符合条件的场景目录：{SPLIT_DIR}")

best = None
for candidate_scene_dir in scene_dirs:
    gt_path = candidate_scene_dir / "scene_gt.json"
    info_path = candidate_scene_dir / "scene_gt_info.json"
    if not gt_path.exists() or not info_path.exists():
        continue

    candidate_gt = read_json(gt_path)
    candidate_info = read_json(info_path)
    for image_key, instances in candidate_gt.items():
        for candidate_gt_id, instance in enumerate(instances):
            visibility = float(
                candidate_info[image_key][candidate_gt_id].get("visib_fract", 0.0)
            )
            if best is None or visibility > best[0]:
                best = (
                    visibility,
                    candidate_scene_dir,
                    int(image_key),
                    candidate_gt_id,
                    instance,
                )

if best is None:
    raise ValueError("所选场景范围内没有可用的实例标注。")

visibility, scene_dir, image_id, gt_id, gt = best
scene_camera = read_json(scene_dir / "scene_camera.json")
camera = scene_camera[str(image_id)]
obj_id = int(gt["obj_id"])

print(f"场景编号: {scene_dir.name}")
print(f"图像编号: {image_id:06d}")
print(f"实例序号 gt_id: {gt_id}")
print(f"模型编号 obj_id: {obj_id}")
print(f"可见比例: {visibility:.3f}")


场景编号: 000048
图像编号: 000036
实例序号 gt_id: 1
模型编号 obj_id: 6
可见比例: 1.000


## 2. 从实例深度区域生成相机坐标点云

使用 `mask_visib` 选择当前实例可见区域，再排除无效深度。BOP 深度值乘以当前帧的 `depth_scale` 后转换为毫米。相机内参将有效像素反投影为三维点。


In [12]:
depth_path = scene_dir / "depth" / f"{image_id:06d}.png"
mask_path = scene_dir / "mask_visib" / f"{image_id:06d}_{gt_id:06d}.png"

depth_raw = np.asarray(o3d.io.read_image(str(depth_path)))
mask_raw = np.asarray(o3d.io.read_image(str(mask_path)))
if depth_raw.ndim == 3:
    depth_raw = depth_raw[:, :, 0]
if mask_raw.ndim == 3:
    mask_raw = mask_raw[:, :, 0]

visible_mask = mask_raw > 0
depth_scale = float(camera["depth_scale"])
depth_mm = depth_raw.astype(np.float64) * depth_scale
valid_mask = visible_mask & np.isfinite(depth_mm) & (depth_raw > 0) & (depth_mm > 0)

if depth_raw.shape != mask_raw.shape:
    raise ValueError("深度图和实例掩码的尺寸不一致。")
if not np.any(valid_mask):
    raise ValueError("当前实例区域内没有有效深度。")

# 读取相机内参矩阵，用于将像素齐次坐标转换为相机射线。
K = np.asarray(camera["cam_K"], dtype=np.float64).reshape(3, 3)
# 读取物体到相机的真值旋转和平移，供后续位姿比较使用。
R_gt = np.asarray(gt["cam_R_m2c"], dtype=np.float64).reshape(3, 3)
t_gt = np.asarray(gt["cam_t_m2c"], dtype=np.float64).reshape(3)

# np.nonzero 按图像数组顺序返回行坐标 v 和列坐标 u。
v_coords, u_coords = np.nonzero(valid_mask)
# 提取这些有效像素对应的深度值，单位为毫米。
z_values = depth_mm[v_coords, u_coords]
# 将像素坐标组成齐次坐标 [u, v, 1]，每一列对应一个有效像素。
pixels_h = np.vstack(
    (u_coords, v_coords, np.ones_like(u_coords))
).astype(np.float64)
# 求解 K @ rays_c = pixels_h，得到相机坐标系中的射线方向。
rays_c = np.linalg.solve(K, pixels_h)
# 射线乘以对应深度后得到三维点；转置为“点数 × 3”的数组。
observed_points_c = (rays_c * z_values[None, :]).T

# 固定随机种子，限制点数，减少迭代和绘图开销。
rng = np.random.default_rng(0)
# 设置观测点数上限，限制后续点云处理的计算量。
max_observed_points = 8000
if len(observed_points_c) > max_observed_points:
    # 无放回随机抽取指定数量的点，避免重复采样。
    selected = rng.choice(
        len(observed_points_c), max_observed_points, replace=False
    )
    # 只保留抽样得到的点。
    observed_points_c = observed_points_c[selected]

# 将 N×3 的 NumPy 坐标数组转换为 Open3D 点云对象。
observed_cloud_c = o3d.geometry.PointCloud()
observed_cloud_c.points = o3d.utility.Vector3dVector(observed_points_c)
# 按体素边长降采样，减少局部密集和冗余点。
observed_cloud_c = observed_cloud_c.voxel_down_sample(VOXEL_SIZE_MM)

print(f"有效深度像素数: {len(v_coords)}")
print(f"ICP 使用的观测点数: {len(observed_cloud_c.points)}")
print(f"深度比例 depth_scale: {depth_scale}")
print(f"内参矩阵 K:\n{K}")


有效深度像素数: 4406
ICP 使用的观测点数: 529
深度比例 depth_scale: 0.1
内参矩阵 K:
[[1.066778e+03 0.000000e+00 3.129869e+02]
 [0.000000e+00 1.067487e+03 2.413109e+02]
 [0.000000e+00 0.000000e+00 1.000000e+00]]


## 3. 从 CAD 网格生成模型点云

CAD 网格处于物体坐标系 O。此处对其表面均匀采样，并使用与观测点云相同的体素尺寸降采样。模型点云将作为 ICP 的目标点云。


In [13]:
model_path = MODELS_DIR / f"obj_{obj_id:06d}.ply"
mesh = o3d.io.read_triangle_mesh(str(model_path))
if mesh.is_empty():
    raise ValueError(f"CAD 网格读取失败或为空：{model_path}")

mesh.compute_triangle_normals()
cad_cloud_o = mesh.sample_points_uniformly(
    number_of_points=CAD_POINT_COUNT,
    use_triangle_normal=True,
)
cad_cloud_o = cad_cloud_o.voxel_down_sample(VOXEL_SIZE_MM)

print(f"CAD 文件: {model_path}")
print(f"ICP 使用的 CAD 点数: {len(cad_cloud_o.points)}")
print("CAD 坐标单位：毫米")


CAD 文件: /Users/liuyuan/projects/具身智能中心/embodied-intelligence/.local/data/bop/ycbv/models/obj_000006.ply
ICP 使用的 CAD 点数: 2459
CAD 坐标单位：毫米


## 4. 用 PPF 产生粗位姿候选

对有向点对 $(p_i,n_i)$ 与 $(p_j,n_j)$，令 $d=p_j-p_i$，PPF 特征为：

$$
F(p_i,p_j)=\left(\lVert d\rVert,\angle(n_i,d),\angle(n_j,d),\angle(n_i,n_j)\right).
$$

PPF 将模型点对特征量化并存入哈希表；场景点对查询哈希表，并在每个场景参考点的投票空间中累计模型参考点和绕法线的角度。多个场景参考点支持的位姿候选经过聚类后，得到粗位姿 $T_{CO}$。

本实现保留 PPF 的“点对特征查表、参考点与角度投票、位姿候选聚类”流程。为便于在 Notebook 中运行，会对点云和参考点数量设上限。掩码来自 BOP 真值标注；PPF 只接收掩码提取的观测点云和 CAD，不读取真值位姿。


In [14]:
# 步骤 1：准备两组带法线的点云，并统一场景法线的朝向。
# PPF 使用有向点对；观测法线朝向相机，CAD 法线取三角网格法线。
observed_cloud_c.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=PPF_NORMAL_RADIUS_MM, max_nn=30
    )
)
observed_cloud_c.orient_normals_towards_camera_location(np.zeros(3))

rng_ppf = np.random.default_rng(7)

# PPF 点对计算随点数近似按平方增长，因此先对模型和场景点云限量采样。
# 点对计算这一步产生的是局部几何特征F(p_i, p_j)，还没有直接算出完整的六自由度位姿；位姿是在后续匹配投票和聚类中得到的。
def sample_ppf_cloud(cloud, max_points):
    points = np.asarray(cloud.points, dtype=np.float64)
    normals = np.asarray(cloud.normals, dtype=np.float64)
    if len(points) > max_points:
        indices = np.sort(rng_ppf.choice(len(points), max_points, replace=False))
        points = points[indices]
        normals = normals[indices]
    lengths = np.linalg.norm(normals, axis=1, keepdims=True)
    normals = normals / np.maximum(lengths, 1e-12)
    return points, normals

# model_points_ppf：从 CAD 点云 cad_cloud_o 抽取的模型点，位于物体坐标系。
# model_normals_ppf：这些模型点各自对应的表面法线。
model_points_ppf, model_normals_ppf = sample_ppf_cloud(
    cad_cloud_o, PPF_MODEL_POINT_COUNT
)
# scene_points_ppf：从深度图得到的场景点云 observed_cloud_c 抽取的观测点，位于相机坐标系。
# scene_normals_ppf：这些场景点各自对应的法线，前面已从观测点云估计。
scene_points_ppf, scene_normals_ppf = sample_ppf_cloud(
    observed_cloud_c, PPF_SCENE_POINT_COUNT
)

# 将距离和角度连续值量化成离散特征格，便于用哈希表查找匹配。
ppf_angle_step = np.deg2rad(PPF_ANGLE_STEP_DEG)
ppf_angle_bin_count = int(round(2.0 * np.pi / ppf_angle_step))
ppf_angle_step = 2.0 * np.pi / ppf_angle_bin_count
ppf_model_diameter = float(
    np.linalg.norm(np.ptp(model_points_ppf, axis=0))
)

# 将角度折返到 [-pi, pi)，避免跨越边界的角度被当成相距很远。
def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi

# 对一个参考点与其余点构造有向点对特征，并返回特征格编号。
def ppf_feature_bins(points, normals, reference_index):
    # 点对方向统一定义为“参考点指向第二个点”。
    differences = points - points[reference_index]
    distances = np.linalg.norm(differences, axis=1)
    # 去掉过近点对，并限制距离不超过采样模型的直径。
    valid = (
        (distances >= PPF_DISTANCE_STEP_MM * 0.5)
        & (distances <= ppf_model_diameter)
    )
    second_indices = np.flatnonzero(valid)
    directions = differences[second_indices] / distances[second_indices, None]

    reference_normal = normals[reference_index]
    second_normals = normals[second_indices]
    angle_1 = np.arctan2(
        np.linalg.norm(np.cross(reference_normal, directions), axis=1),
        np.sum(reference_normal * directions, axis=1),
    )
    angle_2 = np.arctan2(
        np.linalg.norm(np.cross(second_normals, directions), axis=1),
        np.sum(second_normals * directions, axis=1),
    )
    angle_3 = np.arctan2(
        np.linalg.norm(np.cross(reference_normal, second_normals), axis=1),
        np.sum(reference_normal * second_normals, axis=1),
    )
    # 特征顺序为距离、参考法线角、第二点法线角、两法线夹角。
    bins = np.column_stack((
        np.floor(distances[second_indices] / PPF_DISTANCE_STEP_MM),
        np.floor(angle_1 / ppf_angle_step),
        np.floor(angle_2 / ppf_angle_step),
        np.floor(angle_3 / ppf_angle_step),
    )).astype(np.int16)
    return second_indices, bins

# 建立参考点局部坐标系：参考点移到原点，法线旋到 +X 方向。
def align_reference_normal_to_x(point, normal):
    normal = normal / np.linalg.norm(normal)
    x_axis = np.array([1.0, 0.0, 0.0])
    cosine = float(np.clip(np.dot(normal, x_axis), -1.0, 1.0))
    cross = np.cross(normal, x_axis)
    sine = float(np.linalg.norm(cross))

    if sine < 1e-10:
        rotation = np.eye(3) if cosine > 0.0 else np.diag([-1.0, 1.0, -1.0])
    else:
        skew = np.array([
            [0.0, -cross[2], cross[1]],
            [cross[2], 0.0, -cross[0]],
            [-cross[1], cross[0], 0.0],
        ])
        rotation = (
            np.eye(3) + skew
            + (skew @ skew) * ((1.0 - cosine) / (sine * sine))
        )

    transform = np.eye(4)
    transform[:3, :3] = rotation
    transform[:3, 3] = -rotation @ point
    return transform

# 参考法线对齐到 +X 后，alpha 表示第二点绕 X 轴的方位角。
def ppf_alpha(reference_transform, second_point):
    canonical_point = (
        reference_transform[:3, :3] @ second_point
        + reference_transform[:3, 3]
    )
    if np.hypot(canonical_point[1], canonical_point[2]) < 1e-8:
        return None
    return wrap_angle(-np.arctan2(canonical_point[2], canonical_point[1]))

# 步骤 2：遍历 CAD 点对，建立“量化特征 -> 模型参考点和 alpha”的索引。
model_reference_frames = [
    align_reference_normal_to_x(point, normal)
    for point, normal in zip(model_points_ppf, model_normals_ppf)
]
ppf_model_hash = {}
for model_reference_index in range(len(model_points_ppf)):
    second_indices, feature_bins = ppf_feature_bins(
        model_points_ppf, model_normals_ppf, model_reference_index
    )
    reference_frame = model_reference_frames[model_reference_index]
    for second_index, feature_bin in zip(second_indices, feature_bins):
        alpha_model = ppf_alpha(
            reference_frame, model_points_ppf[second_index]
        )
        if alpha_model is None:
            continue
        feature_key = tuple(int(value) for value in feature_bin)
        ppf_model_hash.setdefault(feature_key, []).append(
            (model_reference_index, alpha_model)
        )

# 用相对旋转矩阵的迹计算两个旋转之间的角距离。
def rotation_distance(rotation_a, rotation_b):
    relative = rotation_a @ rotation_b.T
    cosine = np.clip((np.trace(relative) - 1.0) * 0.5, -1.0, 1.0)
    return float(np.arccos(cosine))

# 步骤 3：抽取有限个场景参考点；每个参考点独立统计一份投票。
scene_reference_count = min(
    PPF_SCENE_REFERENCE_COUNT, len(scene_points_ppf)
)
scene_reference_indices = np.sort(
    rng_ppf.choice(
        len(scene_points_ppf), scene_reference_count, replace=False
    )
)
pose_hypotheses = []

# 将当前场景参考点的法线对齐到 +X，便于比较模型与场景的 alpha。
for scene_reference_index in scene_reference_indices:
    scene_reference_frame = align_reference_normal_to_x(
        scene_points_ppf[scene_reference_index],
        scene_normals_ppf[scene_reference_index],
    )
    # 投票格由模型参考点和绕法线的旋转角共同确定。
    accumulator = {}
    second_indices, feature_bins = ppf_feature_bins(
        scene_points_ppf, scene_normals_ppf, scene_reference_index
    )
    for second_index, feature_bin in zip(second_indices, feature_bins):
        alpha_scene = ppf_alpha(
            scene_reference_frame, scene_points_ppf[second_index]
        )
        if alpha_scene is None:
            continue
        feature_key = tuple(int(value) for value in feature_bin)
        # 同一特征可能对应多个模型点对；限制数量以控制投票开销。
        matching_pairs = ppf_model_hash.get(feature_key, ())
        if len(matching_pairs) > PPF_MAX_MATCHES_PER_FEATURE:
            stride = int(np.ceil(
                len(matching_pairs) / PPF_MAX_MATCHES_PER_FEATURE
            ))
            matching_pairs = matching_pairs[::stride]

        for model_reference_index, alpha_model in matching_pairs:
            alpha = wrap_angle(alpha_model - alpha_scene)
            alpha_bin = int(
                np.floor((alpha + np.pi) / ppf_angle_step)
            ) % ppf_angle_bin_count
            # 每个相符的模型点对，为对应的“模型参考点 + alpha 格”加一票。
            vote_key = (model_reference_index, alpha_bin)
            accumulator[vote_key] = accumulator.get(vote_key, 0) + 1

    # 当前场景参考点只保留票数最高的若干组合，生成位姿假设。
    top_votes = sorted(
        accumulator.items(), key=lambda item: item[1], reverse=True
    )[:PPF_TOP_VOTES_PER_REFERENCE]
    for (model_reference_index, alpha_bin), vote_count in top_votes:
        alpha = -np.pi + (alpha_bin + 0.5) * ppf_angle_step
        cosine, sine = np.cos(alpha), np.sin(alpha)
        rotate_x = np.array([
            [1.0, 0.0, 0.0],
            [0.0, cosine, -sine],
            [0.0, sine, cosine],
        ])
        rotation_transform = np.eye(4)
        rotation_transform[:3, :3] = rotate_x
        # 组合模型、场景参考坐标系及绕 +X 的 alpha 旋转，得到模型到相机的变换。
        pose_model_to_scene = (
            np.linalg.inv(scene_reference_frame)
            @ rotation_transform
            @ model_reference_frames[model_reference_index]
        )
        pose_hypotheses.append((
            vote_count, int(scene_reference_index), pose_model_to_scene
        ))

# 步骤 4：把相近的位姿假设合并，汇总来自不同场景参考点的支持。
if not pose_hypotheses:
    raise RuntimeError(
        "PPF 没有产生位姿候选。可检查点云法线和 PPF 距离/角度量化参数。"
    )

pose_clusters = []
for vote_count, scene_reference_index, pose in sorted(
    pose_hypotheses, key=lambda item: item[0], reverse=True
):
    matched_cluster = None
    for cluster in pose_clusters:
        # 只有平移和旋转都足够接近时，才归入同一候选簇。
        translation_distance = np.linalg.norm(
            pose[:3, 3] - cluster["pose"][:3, 3]
        )
        rotation_gap = rotation_distance(
            pose[:3, :3], cluster["pose"][:3, :3]
        )
        if (
            translation_distance <= PPF_CLUSTER_TRANSLATION_MM
            and rotation_gap <= np.deg2rad(PPF_CLUSTER_ROTATION_DEG)
        ):
            matched_cluster = cluster
            break

    if matched_cluster is None:
        pose_clusters.append({
            "pose": pose.copy(),
            "votes": int(vote_count),
            "reference_indices": {scene_reference_index},
        })
    # 同一个场景参考点对一个候选簇最多贡献一次累计票数。
    elif scene_reference_index not in matched_cluster["reference_indices"]:
        matched_cluster["votes"] += int(vote_count)
        matched_cluster["reference_indices"].add(scene_reference_index)

# 按累计票数排序；票数相同时，优先选择获得更多场景参考点支持的簇。
pose_clusters.sort(
    key=lambda item: (item["votes"], len(item["reference_indices"])),
    reverse=True,
)
# 使用最高票簇中的代表位姿作为粗位姿；这里不对簇内位姿做平均。
T_ppf = pose_clusters[0]["pose"]
ppf_vote_count = pose_clusters[0]["votes"]
ppf_reference_support = len(pose_clusters[0]["reference_indices"])

print(f"PPF 模型点对特征记录数: {sum(len(v) for v in ppf_model_hash.values())}")
print(f"PPF 场景参考点数: {scene_reference_count}")
print(f"PPF 位姿候选簇数: {len(pose_clusters)}")
print(f"PPF 主候选累计投票: {ppf_vote_count}")
print(f"PPF 主候选参考点支持数: {ppf_reference_support}")
print("PPF 输出的物体到相机粗位姿 T_CO:")
print(np.round(T_ppf, 3))


PPF 模型点对特征记录数: 67182
PPF 场景参考点数: 120
PPF 位姿候选簇数: 103
PPF 主候选累计投票: 407
PPF 主候选参考点支持数: 6
PPF 输出的物体到相机粗位姿 T_CO:
[[ 6.98000e-01 -7.40000e-02  7.12000e-01 -3.50600e+00]
 [ 6.78000e-01 -2.53000e-01 -6.91000e-01  6.55220e+01]
 [ 2.31000e-01  9.65000e-01 -1.26000e-01  8.08039e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]


## 5. 对照实验：用真值扰动构造 ICP 初值

此对照单独展示 ICP 对初值的敏感性。它通过扰动真值位姿构造较好的初值，不参与 PPF 候选生成，也不作为主流程的输入。

旋转角度、平移量和 ICP 对应距离阈值定义在准备单元中，可修改参数并比较结果变化。


In [15]:
def axis_angle_rotation(axis, angle_degrees):
    axis = np.asarray(axis, dtype=np.float64)
    axis /= np.linalg.norm(axis)
    angle = np.deg2rad(angle_degrees)

    x, y, z = axis
    skew = np.array([
        [0.0, -z, y],
        [z, 0.0, -x],
        [-y, x, 0.0],
    ])
    return (
        np.eye(3)
        + np.sin(angle) * skew
        + (1.0 - np.cos(angle)) * (skew @ skew)
    )


T_gt = np.eye(4)
T_gt[:3, :3] = R_gt
T_gt[:3, 3] = t_gt

delta_T = np.eye(4)
delta_T[:3, :3] = axis_angle_rotation([1.0, 0.6, 0.3], PERTURB_ROTATION_DEG)
delta_T[:3, 3] = PERTURB_TRANSLATION_MM

# 右乘扰动表示在物体坐标系中施加旋转和平移。
T_init = T_gt @ delta_T

print("真值位姿 T_CO:")
print(np.round(T_gt, 3))
print("\n初始位姿 T_CO（真值加扰动）:")
print(np.round(T_init, 3))


真值位姿 T_CO:
[[-8.86000e-01  4.63000e-01 -3.60000e-02  3.51800e+00]
 [ 1.36000e-01  1.86000e-01 -9.73000e-01  5.37670e+01]
 [-4.44000e-01 -8.67000e-01 -2.28000e-01  8.03299e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]

初始位姿 T_CO（真值加扰动）:
[[-8.72000e-01  4.77000e-01 -1.08000e-01 -3.82600e+00]
 [ 1.82000e-01  1.12000e-01 -9.77000e-01  4.89770e+01]
 [-4.54000e-01 -8.72000e-01 -1.84000e-01  8.02962e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]


## 6. 将 PPF 候选交给 ICP 做局部细化

PPF 输出的 $T_{CO}$ 将模型点从物体坐标系 O 变换到相机坐标系 C。ICP 的 source 是相机坐标系中的深度观测点云，target 是物体坐标系中的 CAD 点云，因此传入 ICP 的初值为 $T_{CO}^{-1}$。ICP 输出 source 到 target 的变换，求逆后得到细化后的物体到相机位姿。

点到点 ICP 迭代执行最近邻匹配和刚体变换更新，直至满足收敛条件或达到迭代次数上限。fitness 表示阈值范围内的 source 对应点比例；inlier_rmse 表示这些匹配点的均方根距离，单位为毫米。随后对照 ICP 从 PPF 候选和真值扰动初值开始的结果。


In [16]:
def run_icp_from_pose(T_co_initial):
    return o3d.pipelines.registration.registration_icp(
        source=observed_cloud_c,
        target=cad_cloud_o,
        max_correspondence_distance=MAX_CORRESPONDENCE_DISTANCE_MM,
        init=np.linalg.inv(T_co_initial),
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=MAX_ICP_ITERATIONS
        ),
    )

# 主流程：PPF 粗位姿 -> ICP 局部细化。
icp_ppf_result = run_icp_from_pose(T_ppf)
T_ppf_icp = np.linalg.inv(icp_ppf_result.transformation)

# 对照流程：真值扰动初值 -> ICP；该初值使用真值，不属于 PPF 估计结果。
icp_control_result = run_icp_from_pose(T_init)
T_control_icp = np.linalg.inv(icp_control_result.transformation)

print(f"PPF -> ICP fitness: {icp_ppf_result.fitness:.4f}")
print(f"PPF -> ICP inlier RMSE (mm): {icp_ppf_result.inlier_rmse:.3f}")
print("PPF -> ICP 细化后的物体到相机位姿 T_CO:")
print(np.round(T_ppf_icp, 3))
print(f"真值扰动初值 -> ICP fitness: {icp_control_result.fitness:.4f}")
print(f"真值扰动初值 -> ICP inlier RMSE (mm): {icp_control_result.inlier_rmse:.3f}")


PPF -> ICP fitness: 1.0000
PPF -> ICP inlier RMSE (mm): 1.872
PPF -> ICP 细化后的物体到相机位姿 T_CO:
[[ 9.92000e-01 -1.22000e-01  2.90000e-02  2.93000e+00]
 [ 1.00000e-03 -2.25000e-01 -9.74000e-01  5.36420e+01]
 [ 1.25000e-01  9.67000e-01 -2.23000e-01  8.03855e+02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]
真值扰动初值 -> ICP fitness: 1.0000
真值扰动初值 -> ICP inlier RMSE (mm): 1.998


## 7. 位姿误差计算

旋转误差通过两个旋转矩阵的测地角计算；平移误差是两个平移向量之间的欧氏距离。此处分别计算 PPF 粗位姿、PPF 后接 ICP、真值扰动初值和对照 ICP 的误差。对称物体可能存在外观等价的旋转，因此单独的旋转角误差不一定反映表面几何差异；BOP 正式评测采用考虑物体对称性的指标。


In [17]:
def pose_error(T_estimate, T_reference):
    rotation_delta = T_estimate[:3, :3] @ T_reference[:3, :3].T
    cosine = (np.trace(rotation_delta) - 1.0) / 2.0
    rotation_degrees = np.rad2deg(
        np.arccos(np.clip(cosine, -1.0, 1.0))
    )
    translation_mm = np.linalg.norm(
        T_estimate[:3, 3] - T_reference[:3, 3]
    )
    return rotation_degrees, translation_mm


ppf_rotation_error, ppf_translation_error = pose_error(T_ppf, T_gt)
ppf_icp_rotation_error, ppf_icp_translation_error = pose_error(T_ppf_icp, T_gt)
control_rotation_error, control_translation_error = pose_error(T_init, T_gt)
control_icp_rotation_error, control_icp_translation_error = pose_error(
    T_control_icp, T_gt
)

print("相对真值的位姿误差")
print(
    f"PPF 粗位姿：旋转 {ppf_rotation_error:.3f}°，"
    f"平移 {ppf_translation_error:.3f} mm"
)
print(
    f"PPF -> ICP：旋转 {ppf_icp_rotation_error:.3f}°，"
    f"平移 {ppf_icp_translation_error:.3f} mm"
)
print(
    f"真值扰动初值：旋转 {control_rotation_error:.3f}°，"
    f"平移 {control_translation_error:.3f} mm"
)
print(
    f"真值扰动初值 -> ICP：旋转 {control_icp_rotation_error:.3f}°，"
    f"平移 {control_icp_translation_error:.3f} mm"
)


相对真值的位姿误差
PPF 粗位姿：旋转 159.263°，平移 14.490 mm
PPF -> ICP：旋转 159.410°，平移 0.819 mm
真值扰动初值：旋转 5.000°，平移 8.775 mm
真值扰动初值 -> ICP：旋转 6.311°，平移 1.954 mm


## 8. 点云对齐可视化

分别使用真值、PPF 粗位姿、PPF 后接 ICP 的位姿和真值扰动对照 ICP 位姿，将相机坐标系中的观测点变换回物体坐标系。各面板均显示 CAD 点（橙色）和深度观测点（蓝色），用于比较候选生成与局部细化的效果。


In [18]:
def transform_points(T, points):
    return (T[:3, :3] @ points.T).T + T[:3, 3]


observed_points = np.asarray(observed_cloud_c.points)
cad_points = np.asarray(cad_cloud_o.points)

poses_to_compare = [
    ("真值位姿", T_gt),
    ("PPF 粗位姿", T_ppf),
    ("PPF -> ICP", T_ppf_icp),
    ("真值扰动初值 -> ICP", T_control_icp),
]

fig = make_subplots(
    rows=1,
    cols=len(poses_to_compare),
    specs=[[{"type": "scene"} for _ in poses_to_compare]],
    subplot_titles=[title for title, _ in poses_to_compare],
)

for col, (_, T_co) in enumerate(poses_to_compare, start=1):
    observed_points_o = transform_points(np.linalg.inv(T_co), observed_points)
    fig.add_trace(go.Scatter3d(
        x=cad_points[:, 0],
        y=cad_points[:, 1],
        z=cad_points[:, 2],
        mode="markers",
        marker=dict(size=2, color="darkorange", opacity=0.25),
        name="CAD 点",
        showlegend=(col == 1),
    ), row=1, col=col)
    fig.add_trace(go.Scatter3d(
        x=observed_points_o[:, 0],
        y=observed_points_o[:, 1],
        z=observed_points_o[:, 2],
        mode="markers",
        marker=dict(size=2, color="royalblue", opacity=0.8),
        name="深度观测点",
        showlegend=(col == 1),
    ), row=1, col=col)

fig.update_layout(
    title=f"场景 {scene_dir.name}，图像 {image_id:06d}，obj_id={obj_id}",
    width=1800,
    height=560,
    margin=dict(l=10, r=10, t=70, b=10),
    scene=dict(xaxis_title="X_O (mm)", yaxis_title="Y_O (mm)", zaxis_title="Z_O (mm)", aspectmode="data"),
    scene2=dict(xaxis_title="X_O (mm)", yaxis_title="Y_O (mm)", zaxis_title="Z_O (mm)", aspectmode="data"),
    scene3=dict(xaxis_title="X_O (mm)", yaxis_title="Y_O (mm)", zaxis_title="Z_O (mm)", aspectmode="data"),
    scene4=dict(xaxis_title="X_O (mm)", yaxis_title="Y_O (mm)", zaxis_title="Z_O (mm)", aspectmode="data"),
    legend=dict(itemsizing="constant"),
)
fig.show()


## 结果范围与参数影响

- 主流程使用 PPF 点对特征匹配和参考点投票产生粗位姿，再将该候选作为 ICP 初值。
- PPF 候选质量受法线方向、点云采样、特征量化、遮挡和几何重复性影响；ICP 可能改善候选，也可能因初值偏差较大而失败。
- fitness 和 inlier RMSE 只描述阈值内的几何对应，不单独证明旋转方向正确。若位姿角误差大但点云重合较好，应检查几何对称或重复结构，并参考 BOP 指标。
- 真值扰动对照用于单独观察 ICP 的局部收敛，不属于 PPF 流程。
- BOP 可见掩码来自真值标注，因此本实验没有评估目标检测或分割。
- 对称物体可能出现旋转角误差较大但表面几何接近的情况；正式比较需使用考虑对称性的 BOP 指标。

### 思考题

1. 为什么 source 点云到 CAD 的 ICP 初值是 $T_{CO}^{-1}$？
2. PPF 的距离步长、角度步长和参考点数量分别会如何影响候选数量与稳定性？
3. ICP 从 PPF 候选和真值扰动初值开始时，结果有什么差别？这说明什么？
4. 为什么使用真值掩码，仍不能证明完整场景中的目标检测和位姿估计性能？
5. 对称物体上，为什么旋转误差可能较大而几何对齐仍然合理？
